# 03 — Write to Delta

Persist each sheet as a managed Delta table in `alexander_booth.salt_river_excel`.
Applies proper type casting after the initial string-safe load.

In [1]:
from databricks.connect import DatabricksSession
from dotenv import load_dotenv
import os

load_dotenv()

spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA  = os.getenv("UC_SCHEMA",  "salt_river_excel")
UC_VOLUME  = os.getenv("UC_VOLUME",  "raw_files")

FILE_PATH = f"/Volumes/{UC_CATALOG}/{UC_SCHEMA}/{UC_VOLUME}/salt_river_fields.xlsx"

In [2]:
from pyspark.sql import functions as F

# ── Attendance ──────────────────────────────────────────────────────────────
attendance_raw = (
    spark.read
    .option("dataAddress", "Attendance")
    .option("headerRows", 1)
    .excel(FILE_PATH)
)

attendance = attendance_raw.select(
    F.col("game_date").cast("date").alias("game_date"),
    F.col("home_team"),
    F.col("opponent"),
    F.col("attendance").cast("int").alias("attendance"),
    F.col("capacity").cast("int").alias("capacity"),
    F.col("pct_full").cast("double").alias("pct_full"),
    F.col("gate_revenue_usd").cast("double").alias("gate_revenue_usd"),
)

attendance.write.format("delta").mode("overwrite").option("overwriteSchema", True) \
    .saveAsTable(f"{UC_CATALOG}.{UC_SCHEMA}.attendance")

print(f"attendance: {attendance.count()} rows written")

attendance: 18 rows written


In [3]:
# ── Concessions ─────────────────────────────────────────────────────────────
concessions_raw = (
    spark.read
    .option("dataAddress", "Concessions")
    .option("headerRows", 1)
    .excel(FILE_PATH)
)

concessions = concessions_raw.select(
    F.col("game_date").cast("date").alias("game_date"),
    F.col("home_team"),
    F.col("item_name"),
    F.col("category"),
    F.col("unit_price_usd").cast("double").alias("unit_price_usd"),
    F.col("units_sold").cast("int").alias("units_sold"),
    F.col("total_revenue_usd").cast("double").alias("total_revenue_usd"),
)

concessions.write.format("delta").mode("overwrite").option("overwriteSchema", True) \
    .saveAsTable(f"{UC_CATALOG}.{UC_SCHEMA}.concessions")

print(f"concessions: {concessions.count()} rows written")

concessions: 270 rows written


In [4]:
# ── Merchandise ─────────────────────────────────────────────────────────────
merch_raw = (
    spark.read
    .option("dataAddress", "Merchandise")
    .option("headerRows", 1)
    .excel(FILE_PATH)
)

merch = merch_raw.select(
    F.col("game_date").cast("date").alias("game_date"),
    F.col("home_team"),
    F.col("sku"),
    F.col("item_name"),
    F.col("category"),
    F.col("team"),
    F.col("unit_price_usd").cast("double").alias("unit_price_usd"),
    F.col("units_sold").cast("int").alias("units_sold"),
    F.col("total_revenue_usd").cast("double").alias("total_revenue_usd"),
)

merch.write.format("delta").mode("overwrite").option("overwriteSchema", True) \
    .saveAsTable(f"{UC_CATALOG}.{UC_SCHEMA}.merchandise")

print(f"merchandise: {merch.count()} rows written")

merchandise: 342 rows written


In [5]:
# ── Verify tables ───────────────────────────────────────────────────────────
spark.sql(f"SHOW TABLES IN {UC_CATALOG}.{UC_SCHEMA}").show()

# Revenue summary across all streams per game
spark.sql(f"""
    SELECT
        a.game_date,
        a.home_team,
        a.opponent,
        a.attendance,
        ROUND(a.gate_revenue_usd, 2)                       AS gate,
        ROUND(SUM(c.total_revenue_usd), 2)                 AS concessions,
        ROUND(SUM(m.total_revenue_usd), 2)                 AS merch,
        ROUND(a.gate_revenue_usd
              + SUM(c.total_revenue_usd)
              + SUM(m.total_revenue_usd), 2)               AS total_revenue
    FROM {UC_CATALOG}.{UC_SCHEMA}.attendance a
    LEFT JOIN {UC_CATALOG}.{UC_SCHEMA}.concessions c USING (game_date)
    LEFT JOIN {UC_CATALOG}.{UC_SCHEMA}.merchandise  m USING (game_date)
    GROUP BY a.game_date, a.home_team, a.opponent, a.attendance, a.gate_revenue_usd
    ORDER BY a.game_date
""").show(20, truncate=False)

+----------------+-----------+-----------+
|        database|  tableName|isTemporary|
+----------------+-----------+-----------+
|salt_river_excel| attendance|      false|
|salt_river_excel|concessions|      false|
|salt_river_excel|merchandise|      false|
+----------------+-----------+-----------+

+----------+--------------------+---------+----------+---------+-----------+--------+-------------+
|game_date |home_team           |opponent |attendance|gate     |concessions|merch   |total_revenue|
+----------+--------------------+---------+----------+---------+-----------+--------+-------------+
|2025-02-22|Colorado Rockies    |Padres   |9819      |187673.6 |547865.0   |432330.0|1167868.6    |
|2025-02-24|Arizona Diamondbacks|Cubs     |10237     |212420.75|530489.5   |406905.0|1149815.25   |
|2025-02-26|Colorado Rockies    |Brewers  |8114      |157374.11|603696.5   |430245.0|1191315.61   |
|2025-02-28|Arizona Diamondbacks|Padres   |7619      |188699.73|603231.0   |343890.0|1135820.73   